> **MIT License** - Copyright (c) 2026 QuantWall. Full text: https://opensource.org/license/mit
>
> **Disclaimer.** This content is general information and software-engineering education only. It is not financial product advice, a recommendation, or an offer to buy or sell any financial product, and it does not take into account your objectives, financial situation or needs. The author does not hold an Australian Financial Services Licence and is not a registered Commodity Trading Advisor. Trading futures and leveraged products carries a high risk of loss. Consult a licensed adviser before making any financial decision.
>
> **HYPOTHETICAL OR SIMULATED PERFORMANCE RESULTS HAVE CERTAIN LIMITATIONS. UNLIKE AN ACTUAL PERFORMANCE RECORD, SIMULATED RESULTS DO NOT REPRESENT ACTUAL TRADING. ALSO, SINCE THE TRADES HAVE NOT BEEN EXECUTED, THE RESULTS MAY HAVE UNDER-OR-OVER COMPENSATED FOR THE IMPACT, IF ANY, OF CERTAIN MARKET FACTORS, SUCH AS LACK OF LIQUIDITY. SIMULATED TRADING PROGRAMS IN GENERAL ARE ALSO SUBJECT TO THE FACT THAT THEY ARE DESIGNED WITH THE BENEFIT OF HINDSIGHT. NO REPRESENTATION IS BEING MADE THAT ANY ACCOUNT WILL OR IS LIKELY TO ACHIEVE PROFIT OR LOSSES SIMILAR TO THOSE SHOWN.**
>
> All data shown is synthetic or publicly available sample data. Features are standard textbook baselines (lagged returns, ATR, RSI, moving averages, volume). No proprietary signal is disclosed or implied.

# What the wall is: every number in the video

One real set of numbers (the trial counts, from Kaplan & Irvin, PLoS ONE, 2015). Everything else is illustrative: the gauge sizes, the barn, the 1,000 made-up ideas (100 real is an assumption), and the outcome axis. Same code as the animations (`animations_manim/wall_sim.py`).

In [ ]:
import numpy as np

import numpy as np

# REAL: large NHLBI-funded trials of drugs or supplements, 1970-2012, reporting a significant benefit on the
# primary outcome; all trials published after 2000 were pre-registered (ClinicalTrials.gov). An association:
# the authors say the trend cannot be shown to be caused by registration.
TRIALS = {"before": (17, 30), "after": (2, 25), "year": 2000,
          "source": "Kaplan RM, Irvin VL (2015). Likelihood of Null Effects of Large NHLBI Clinical Trials Has Increased over Time. PLoS ONE 10(8): e0132382."}


def share(k_n):
    k, n = k_n
    return k / n


# ---------------------------------------------------------------------------------------------
# The gauge: a hole's limits on a size ruler (illustrative millimetres) and three parts
# ---------------------------------------------------------------------------------------------
GAUGE = {"low": 20.00, "high": 20.05, "go": 20.00, "nogo": 20.06,
         "parts": [("in limits", 20.03), ("too small", 19.97), ("too big", 20.08)]}


def gauge_verdict(size, nogo_present=True):
    go_enters = size >= GAUGE["go"]
    nogo_enters = size >= GAUGE["nogo"]
    if not go_enters:
        return "no: too small"
    if nogo_present and nogo_enters:
        return "no: too big"
    return "pass"


# ---------------------------------------------------------------------------------------------
# The barn: random holes, a target painted after vs before
# ---------------------------------------------------------------------------------------------
def barn(seed=8, n=36, w=10.0, h=4.0, r=0.9):
    rng = np.random.default_rng(seed)
    holes = np.c_[rng.uniform(0, w, n), rng.uniform(0, h, n)]
    # painted after: the circle of radius r holding the most holes, centred on a hole (the tightest bunch)
    counts = [(np.hypot(*(holes - c).T) <= r).sum() for c in holes]
    after = holes[int(np.argmax(counts))]
    # painted first: a fixed centre chosen before any shot
    first = np.array([w * 0.5, h * 0.5])
    hits_first = int((np.hypot(*(holes - first).T) <= r).sum())
    return {"holes": holes, "after": after, "hits_after": int(max(counts)), "first": first, "hits_first": hits_first,
            "w": w, "h": h, "r": r}


# ---------------------------------------------------------------------------------------------
# What a yes is worth: 1,000 made-up ideas
# ---------------------------------------------------------------------------------------------
IDEAS = {"n": 1000, "real": 100, "fake_pass": 1 / 20, "real_pass": 1 / 2}


def ideas(seed=3, cols=40):
    rng = np.random.default_rng(seed)
    n, k = IDEAS["n"], IDEAS["real"]
    real = np.zeros(n, bool)
    real[rng.choice(n, k, replace=False)] = True
    fake_idx, real_idx = np.nonzero(~real)[0], np.nonzero(real)[0]
    passed = np.zeros(n, bool)
    passed[rng.choice(fake_idx, round(len(fake_idx) * IDEAS["fake_pass"]), replace=False)] = True
    passed[rng.choice(real_idx, round(len(real_idx) * IDEAS["real_pass"]), replace=False)] = True
    yes_real, yes_fake = int((passed & real).sum()), int((passed & ~real).sum())
    return {"real": real, "passed": passed, "cols": cols, "rows": n // cols,
            "yes_real": yes_real, "yes_fake": yes_fake,
            "share_real_wall": yes_real / (yes_real + yes_fake), "share_real_stamp": k / n,
            "rejected_real": int((~passed & real).sum()), "rejected": int((~passed).sum())}


# ---------------------------------------------------------------------------------------------
# A door in the wall: one noise idea, retried until it lands in the yes zone (illustrative axis)
# ---------------------------------------------------------------------------------------------
LINE = 1.645           # the yes zone starts here on the made-up axis


def door_walk(seed=None, max_tries=40):
    """Draws for one idea with no edge, one per 'unless', until one lands past the line."""
    if seed is None:
        for s in range(500):           # a walk of 6-8 tries: long enough to see, short enough to draw
            d = door_walk(s)
            if 6 <= len(d) <= 8:
                return d
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(max_tries):
        x = float(rng.normal())
        out.append(x)
        if x > LINE:
            break
    return out

## The trials (real)

In [ ]:
for k in ('before', 'after'):
    a, n = TRIALS[k]
    print(f'{k} {TRIALS["year"]}: {a} of {n} = {a / n:.0%}')
print(TRIALS['source'])

## The gauge
A part passes only if the GO end enters and the NO-GO end does not. Grind the NO-GO end away and oversized parts pass.

In [ ]:
for name, size in GAUGE['parts']:
    print(f'{name:10s} {size:.2f} mm   with NO-GO: {gauge_verdict(size):14s} without: {gauge_verdict(size, False)}')

## The barn
A target painted around the tightest bunch, after the shots, against one painted first.

In [ ]:
b = barn()
print(f"{len(b['holes'])} random holes; inside a target painted after: {b['hits_after']}; painted first: {b['hits_first']}")
print('over 200 barns, painted after vs first (average holes inside):',
      np.mean([barn(s)['hits_after'] for s in range(200)]), np.mean([barn(s)['hits_first'] for s in range(200)]))

## What a yes is worth
The share of real ideas among the yeses, for a rubber stamp and for a wall that lets through 1 fake in 20 and catches half the real ones. Exact arithmetic, then the seeded grid used on screen.

In [ ]:
n, k = IDEAS['n'], IDEAS['real']
fake_yes, real_yes = (n - k) * IDEAS['fake_pass'], k * IDEAS['real_pass']
print(f'rubber stamp: {k / n:.0%} real among yeses')
print(f'wall: {fake_yes:.0f} fake + {real_yes:.0f} real = {real_yes / (real_yes + fake_yes):.1%} real among yeses; real turned away {k - real_yes:.0f}')
i = ideas()
print('grid on screen:', {key: i[key] for key in ('yes_fake', 'yes_real', 'share_real_wall', 'rejected_real')})

## A door in the wall
One idea with no edge, retried until it lands past the line (the axis is made up). With no limit on retries, every idea eventually gets through.

In [ ]:
walk = door_walk()
print([round(x, 2) for x in walk], '-> yes on try', len(walk))
through = [door_walk(s)[-1] > LINE for s in range(10000)]
print('share of 10,000 no-edge ideas that get through within 40 tries:', np.mean(through))